<a href="https://colab.research.google.com/github/eduardobbastos/colabs/blob/main/Transcri%C3%A7%C3%A3o_Audio_Video_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎧📝 Transcrição automática de áudio ou vídeo no Google Colab

Bem-vindo! Este notebook permite **extrair a transcrição de qualquer áudio ou vídeo**, seja por upload de arquivo ou por URL (YouTube, Vimeo, Spotify, etc.), usando o modelo Whisper da OpenAI.

## **Como usar este notebook**

1. **Execute a célula de instalação de pacotes:**  
   Clique no botão de “play” (▶️) na primeira célula de código para instalar as dependências necessárias.  
   _Aguarde até aparecer “Successfully installed...” antes de seguir para as próximas células._

2. **Rode a célula principal do código:**  
   - Escolha o que você deseja transcrever:
     - **1** para vídeo (áudio será extraído automaticamente)
     - **2** para áudio (direto do arquivo ou da URL)
   - Depois, escolha entre fazer upload do arquivo ou informar a URL.
     - **Upload:** selecione o arquivo do seu computador.
     - **URL:** cole o link do vídeo ou do áudio (ex: do YouTube, Vimeo, ou link direto de áudio .mp3).
   - O notebook irá baixar/processar seu arquivo, extrair o áudio se for vídeo e gerar a transcrição.

3. **Aguarde a transcrição:**  
   O tempo de processamento depende do tamanho do arquivo e do modelo usado (por padrão, o modelo é `medium`, com boa fidelidade).

4. **Baixe o arquivo de transcrição:**  
   Após o processamento, um arquivo `.txt` com o texto transcrito será gerado e disponibilizado para download.

---

> **Dica:**  
> Para máxima qualidade, utilize arquivos com boa clareza de áudio.  
> Para arquivos longos ou para máxima precisão, você pode alterar o modelo para `"large"` no código, mas pode demorar mais e consumir mais memória.

---

Pronto! Agora, basta seguir as instruções no notebook e processar seus áudios e vídeos para obter a transcrição em texto, pronta para ser usada em qualquer aplicação de IA ou automação.



In [1]:
!pip install git+https://github.com/openai/whisper.git
!pip install yt-dlp moviepy openai-whisper deep-translator
!sudo apt update && sudo apt install ffmpeg


  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-bpw23s2m
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-bpw23s2m
  Resolved https://github.com/openai/whisper.git to commit c0d2f624c09dc18e709e37c2ad90c039a4eb72a2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=cbda3400abdc38047dcc34a4723c9268e592abd2d49a3cff54dba195d0bf079b
  Stored in directory: /tmp/pip-ephem-wheel-cache-e2ft7k6x/wheels/c3/03/25/5e0ba78bc27a3a089f137c9f1d92fdfce16d06996c071a016c
Successfully built openai-whisper
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 5.4 MB/s eta 0:00:00


In [4]:
# @title 🚀 Transcritor e Tradutor Otimizado (Whisper + Checkpoint)
# @markdown Execute esta célula para instalar as dependências e rodar o programa.

# 1. Instalação Silenciosa
import os
print("📦 Verificando e instalando bibliotecas...")
os.system("pip install -q yt-dlp openai-whisper deep-translator tqdm")

# 2. Imports
import yt_dlp
import json
import time
import re
import torch
import whisper
from google.colab import files
from datetime import datetime, timedelta
from deep_translator import GoogleTranslator
from tqdm import tqdm # Barra de progresso profissional

# 3. Configurações Globais
LANGUAGES = {
    '0': {'name': '🤖 Automático / Misto (Detectar)', 'code': None},
    '1': {'name': '🇧🇷 Português', 'code': 'pt'},
    '2': {'name': '🇺🇸 Inglês', 'code': 'en'},
    '3': {'name': '🇪🇸 Espanhol', 'code': 'es'},
    '4': {'name': '🇫🇷 Francês', 'code': 'fr'},
    '5': {'name': '🇩🇪 Alemão', 'code': 'de'},
    '6': {'name': '🇮🇹 Italiano', 'code': 'it'},
    '7': {'name': '🇳🇱 Holandês', 'code': 'nl'},
    '8': {'name': '🇷🇺 Russo', 'code': 'ru'},
    '9': {'name': '🇵🇱 Polonês', 'code': 'pl'},
    '10': {'name': '🇸🇪 Sueco', 'code': 'sv'}
}

# --- Funções Utilitárias ---

def check_gpu():
    if not torch.cuda.is_available():
        print("\n⚠️ AVISO: Aceleração de Hardware (GPU) NÃO detectada!")
        print("Para ser rápido, vá em: Ambiente de Execução > Alterar tipo > T4 GPU.\n")
    else:
        print(f"\n✅ GPU Detectada: {torch.cuda.get_device_name(0)}")

def sanitize_filename(name):
    """Remove caracteres especiais e espaços para evitar erros de sistema"""
    name = os.path.splitext(name)[0]
    # Remove tudo que não for letra, número, underscore ou traço
    clean = re.sub(r'[^\w\-]', '_', name)
    return clean

def get_timestamp():
    return datetime.now().strftime('%Y%m%d_%H%M%S')

def format_timestamp(seconds):
    """Formata segundos para SRT (HH:MM:SS,mmm)"""
    td = timedelta(seconds=seconds)
    total_seconds = int(td.total_seconds())
    micros = td.microseconds
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    millis = int(micros / 1000)
    return f"{hours:02}:{minutes:02}:{seconds:02},{millis:03}"

def select_menu(options, title):
    print(f"\n--- {title} ---")
    keys = sorted(options.keys(), key=lambda x: int(x))
    for key in keys:
        print(f"{key} - {options[key]['name']}")
    while True:
        choice = input("👉 Selecione o número: ").strip()
        if choice in options:
            return options[choice]
        print("❌ Opção inválida.")

# --- Funções de Download Unificadas ---

def download_media(url, is_audio_only=False):
    """Baixa e já converte para áudio se necessário, usando yt-dlp puro"""
    timestamp = get_timestamp()
    output_base = f"download_{timestamp}"

    # Configuração base
    ydl_opts = {
        'outtmpl': f'{output_base}.%(ext)s',
        'quiet': True,
        'no_warnings': True,
    }

    if is_audio_only:
        # Se for vídeo mas o usuário quer só áudio, ou se for link de música
        ydl_opts.update({
            'format': 'bestaudio/best',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': '192',
            }],
        })
    else:
        # Baixa vídeo MP4 e extrai áudio MP3 separadamente para o Whisper
        ydl_opts.update({
            'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
        })

    print(f"⬇️ Baixando conteúdo de: {url}...")
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            filename = ydl.prepare_filename(info)

            if is_audio_only:
                # Se pediu conversão, o arquivo final é .mp3
                final_name = os.path.splitext(filename)[0] + '.mp3'
                return final_name, final_name
            else:
                # Se baixou vídeo, precisamos extrair o áudio para o Whisper
                video_name = filename
                audio_name = os.path.splitext(filename)[0] + '.mp3'

                # Extração rápida via comando de sistema (mais leve que moviepy)
                if not os.path.exists(audio_name):
                    print("Extracting audio for AI processing...")
                    os.system(f'ffmpeg -i "{video_name}" -q:a 0 -map a "{audio_name}" -y -hide_banner -loglevel error')

                return video_name, audio_name

    except Exception as e:
        print(f"❌ Erro no download: {e}")
        return None, None

# --- FLUXO PRINCIPAL ---

def main():
    check_gpu()
    print('\n=== 🎧 SISTEMA DE TRANSCRIÇÃO E TRADUÇÃO PRO ===')

    # 1. Configuração do Modelo
    print("\n[1] Escolha a Potência da IA:")
    print("1 - Medium (Rápido, consome ~5GB VRAM)")
    print("2 - Large (Preciso, consome ~10GB VRAM - Obrigatório para áudios mistos)")
    model_choice = input("👉 Escolha (1 ou 2): ").strip()
    model_name = "large" if model_choice == '2' else "medium"

    # 2. Fonte de Dados
    print("\n[2] Fonte do Arquivo:")
    print("1 - Link da Internet (YouTube, etc)")
    print("2 - Upload de Arquivo Local")
    source_choice = input("👉 Escolha (1 ou 2): ").strip()

    file_path = ""
    audio_for_ai = "" # Caminho do áudio que o Whisper vai ler
    clean_name = ""

    if source_choice == '1':
        url = input("🔗 Cole a URL: ").strip()
        # Baixa vídeo completo para ter referência, mas extrai áudio para IA
        # Se quiser apenas áudio do youtube, mude is_audio_only para True
        orig_file, audio_file = download_media(url, is_audio_only=False)
        if not orig_file: return
        file_path = orig_file
        audio_for_ai = audio_file
        clean_name = sanitize_filename(orig_file)

    elif source_choice == '2':
        print("⬆️ Faça o upload do arquivo agora...")
        uploaded = files.upload()
        if not uploaded: return
        f_name = list(uploaded.keys())[0]

        # Sanitizar nome logo na entrada
        clean_name = sanitize_filename(f_name)
        ext = os.path.splitext(f_name)[1]
        file_path = f"{clean_name}{ext}"
        os.rename(f_name, file_path)

        # Preparar áudio para IA
        audio_for_ai = f"{clean_name}.mp3"
        # Se já não for mp3/wav, converte
        if ext.lower() not in ['.mp3', '.wav', '.m4a']:
             os.system(f'ffmpeg -i "{file_path}" -q:a 0 -map a "{audio_for_ai}" -y -hide_banner -loglevel error')
        else:
             audio_for_ai = file_path # Já é áudio

    # 3. Checkpoint Check (Antes de perguntar idiomas, verifica se já existe)
    json_checkpoint = f"{clean_name}_checkpoint.json"
    whisper_result = None

    if os.path.exists(json_checkpoint):
        print(f"\n💾 Checkpoint encontrado: {json_checkpoint}")
        use_chk = input("Deseja pular a transcrição e usar os dados salvos? (s/n): ").lower()
        if use_chk == 's':
            with open(json_checkpoint, 'r', encoding='utf-8') as f:
                whisper_result = json.load(f)

    # 4. Configuração de Idiomas (Se não recuperou checkpoint ou para tradução)
    input_lang_cfg = select_menu(LANGUAGES, "Idioma Falado (Entrada)")

    print("\n--- DECISÃO DE TRADUÇÃO ---")
    print("1 - Apenas Transcrever (Original)")
    print("2 - Traduzir para outro idioma")
    trans_dec = input("👉 Escolha (1 ou 2): ").strip()

    target_code = None
    if trans_dec == '2':
        out_lang_cfg = select_menu(LANGUAGES, "Idioma da Legenda (Saída)")
        target_code = out_lang_cfg['code']

    # 5. Execução do Whisper (Se não recuperado)
    if not whisper_result:
        print(f"\n🧠 Carregando modelo Whisper ({model_name})...")
        try:
            model = whisper.load_model(model_name)
        except Exception as e:
            print(f"Erro ao carregar modelo: {e}. Tentando fallback para 'medium'.")
            model = whisper.load_model("medium")

        print("🎙️ Iniciando transcrição... (Isso pode demorar)")
        # fp16=False evita warning em CPU, mas em GPU T4 é seguro usar True.
        # Deixando False para compatibilidade máxima.
        transcribe_opts = {"fp16": False}
        if input_lang_cfg['code']:
            transcribe_opts["language"] = input_lang_cfg['code']

        whisper_result = model.transcribe(audio_for_ai, **transcribe_opts)

        # Salvar Checkpoint
        with open(json_checkpoint, 'w', encoding='utf-8') as f:
            json.dump(whisper_result, f, ensure_ascii=False)
        print("✅ Transcrição concluída e salva.")

    # 6. Processamento e Tradução
    detected_lang = whisper_result.get('language', 'unknown')
    print(f"\n🗣️ Idioma detectado: {detected_lang.upper()}")

    translator = None
    need_translation = False

    if target_code and target_code != detected_lang:
        print(f"🌍 Iniciando tradução: {detected_lang.upper()} -> {target_code.upper()}")
        translator = GoogleTranslator(source='auto', target=target_code)
        need_translation = True

    segments = whisper_result['segments']
    final_txt = ""
    final_srt = ""

    # Barra de progresso
    print("\n📝 Gerando arquivos...")
    for i, segment in enumerate(tqdm(segments, desc="Processando linhas")):
        start = format_timestamp(segment['start'])
        end = format_timestamp(segment['end'])
        text = segment['text'].strip()

        processed = text

        if need_translation and text:
            try:
                translated = translator.translate(text)
                if translated: processed = translated
            except:
                pass # Falha silenciosa mantém original

        final_txt += processed + " "
        final_srt += f"{i+1}\n{start} --> {end}\n{processed}\n\n"

    # 7. Salvar e Download
    suffix = f"_{target_code}" if need_translation else f"_{detected_lang}"
    path_txt = f"{clean_name}{suffix}.txt"
    path_srt = f"{clean_name}{suffix}.srt"

    with open(path_txt, "w", encoding="utf-8") as f: f.write(final_txt.strip())
    with open(path_srt, "w", encoding="utf-8") as f: f.write(final_srt)

    print("\n🎉 Processo Finalizado!")
    files.download(path_txt)
    files.download(path_srt)

if __name__ == "__main__":
    main()

📦 Verificando e instalando bibliotecas...

✅ GPU Detectada: Tesla T4

=== 🎧 SISTEMA DE TRANSCRIÇÃO E TRADUÇÃO PRO ===

[1] Escolha a Potência da IA:
1 - Medium (Rápido, consome ~5GB VRAM)
2 - Large (Preciso, consome ~10GB VRAM - Obrigatório para áudios mistos)
👉 Escolha (1 ou 2): 1

[2] Fonte do Arquivo:
1 - Link da Internet (YouTube, etc)
2 - Upload de Arquivo Local
👉 Escolha (1 ou 2): 1
🔗 Cole a URL: https://www.youtube.com/watch?v=ajsmd0ybOrY
⬇️ Baixando conteúdo de: https://www.youtube.com/watch?v=ajsmd0ybOrY...
Extracting audio for AI processing...

--- Idioma Falado (Entrada) ---
0 - 🤖 Automático / Misto (Detectar)
1 - 🇧🇷 Português
2 - 🇺🇸 Inglês
3 - 🇪🇸 Espanhol
4 - 🇫🇷 Francês
5 - 🇩🇪 Alemão
6 - 🇮🇹 Italiano
7 - 🇳🇱 Holandês
8 - 🇷🇺 Russo
9 - 🇵🇱 Polonês
10 - 🇸🇪 Sueco
👉 Selecione o número: 0

--- DECISÃO DE TRADUÇÃO ---
1 - Apenas Transcrever (Original)
2 - Traduzir para outro idioma
👉 Escolha (1 ou 2): 1

🧠 Carregando modelo Whisper (medium)...
🎙️ Iniciando transcrição... (Isso pode d

Processando linhas: 100%|██████████| 301/301 [00:00<00:00, 97905.04it/s]


🎉 Processo Finalizado!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>